In [1]:
import os
%pwd

'd:\\omar\\MLOps\\Chest-Cancer-Classification\\research'

In [2]:
os.chdir("../")
%pwd

'd:\\omar\\MLOps\\Chest-Cancer-Classification'

In [3]:

import os
import mlflow
import dagshub
import tensorflow as tf

# ===============================
# 2. DagsHub Integration
# ===============================
dagshub.init(
    repo_owner="omarhatem44",
    repo_name="End-to-end-Chest-Cancer-Classification",
    mlflow=True
)
mlflow.set_experiment("Evaluation-Pipeline")

Accessing as omarhatem44

Initialized MLflow to track repo "omarhatem44/End-to-end-Chest-Cancer-Classification"

Repository omarhatem44/End-to-end-Chest-Cancer-Classification initialized!

<Experiment: artifact_location='mlflow-artifacts:/dd4649ccb1ba42ceb57b01f0d82e70a9', creation_time=1774407941501, experiment_id='2', last_update_time=1774407941501, lifecycle_stage='active', name='Evaluation-Pipeline', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [4]:
import tensorflow as tf
model = tf.keras.models.load_model(r"D:\omar\MLOps\Chest-Cancer-Classification\artifacts\training\model.h5")

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [7]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [8]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/Chest-CT-Scan-data",
            mlflow_uri="https://dagshub.com/omarhatem44/End-to-end-Chest-Cancer-Classification.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [ ]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    
    def log_into_mlflow(self):

        with mlflow.start_run():

            # 🔥 tags (اختياري بس مهم للتنظيم)
            mlflow.set_tag("platform", "DagsHub")
            mlflow.set_tag("stage", "evaluation")

            # ✅ Log params
            mlflow.log_params(self.config.all_params)

            # ✅ Log metrics
            mlflow.log_metrics({
                "loss": self.score[0],
                "accuracy": self.score[1]
            })

            # ✅ Log model + register
            mlflow.keras.log_model(self.model,
                artifact_path="model",
                registered_model_name="VGG16Model"
            )

In [10]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

[2026-03-25 07:50:11,461: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-25 07:50:11,468: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-25 07:50:11,470: INFO: common: created directory at: artifacts]


Found 102 images belonging to 2 classes.
7/7 [==============================] - 28s 4s/step - loss: 20.6607 - accuracy: 0.5686
[2026-03-25 07:50:40,865: INFO: common: json file saved at: scores.json]


2026/03/25 07:50:42 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-03-25 07:50:44,088: WARNING: save: Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 14). These functions will not be directly callable after loading.]
INFO:tensorflow:Assets written to: C:\Users\Smart\AppData\Local\Temp\tmp3l9dnrsk\model\data\model\assets
[2026-03-25 07:50:45,028: INFO: builder_impl: Assets written to: C:\Users\Smart\AppData\Local\Temp\tmp3l9dnrsk\model\data\model\assets]


Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/03/25 07:55:56 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: VGG16Model, version 9
Created version '9' of model 'VGG16Model'.
